# Phase 1 — Understanding Embeddings

**Goal:** Get an intuition for what embeddings are and why they power RAG.

By the end of this notebook you will:
- Convert text → vectors using a pre-trained model
- Measure similarity between health-related sentences
- Visualise the embedding space with PCA
- See why "high protein foods" and "best protein sources" end up close together

In [ ]:
# Install if needed (run once)
# !pip install sentence-transformers matplotlib scikit-learn

In [ ]:
import sys
sys.path.append('..')  # so we can import from the project root

from sentence_transformers import SentenceTransformer
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded. Embedding dimension:', model.get_sentence_embedding_dimension())

## 1. Embed some sentences and compare similarity

In [ ]:
sentences = [
    # Nutrition - protein
    "What foods are high in protein?",
    "Best protein sources for muscle building",
    "High protein foods list",
    # Nutrition - iron
    "Iron rich foods to prevent anemia",
    "Foods that contain a lot of iron",
    # Exercise
    "Best exercises for building muscle",
    "How to do strength training",
    "Cardio workout for weight loss",
    # Unrelated
    "How to fix a leaking tap",
    "Python programming tutorial",
]

embeddings = model.encode(sentences, normalize_embeddings=True)
print(f'Shape: {embeddings.shape}  →  {len(sentences)} sentences × {embeddings.shape[1]} dims')

In [ ]:
# Cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

short_labels = [s[:35] + '...' if len(s) > 35 else s for s in sentences]
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)
ax.set_title('Sentence Similarity Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print('\nObservation: nutrition sentences cluster together, exercise sentences cluster together,')
print('and unrelated sentences (tap, Python) score low similarity with everything health-related.')

## 2. Find the most similar sentence to a query

In [ ]:
query = "I want to eat more protein"
query_emb = model.encode([query], normalize_embeddings=True)

scores = cosine_similarity(query_emb, embeddings)[0]
ranked = sorted(zip(sentences, scores), key=lambda x: x[1], reverse=True)

print(f'Query: "{query}"\n')
print('Ranked by similarity:')
for sent, score in ranked:
    bar = '█' * int(score * 20)
    print(f'  {score:.3f} {bar:20s}  {sent}')

## 3. Visualise in 2D with PCA

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

colors = [
    '#2ecc71', '#2ecc71', '#2ecc71',   # nutrition-protein (green)
    '#27ae60', '#27ae60',               # nutrition-iron (dark green)
    '#3498db', '#3498db', '#3498db',    # exercise (blue)
    '#e74c3c', '#e74c3c',               # unrelated (red)
]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=100, zorder=3)

for i, (x, y) in enumerate(coords):
    label = sentences[i][:40]
    ax.annotate(label, (x, y), textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.set_title('Embedding Space (PCA 2D projection)', fontsize=14)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nNotice: similar topics cluster spatially — this is how retrieval finds relevant chunks.')

## 4. Embed a real knowledge chunk and query it

In [ ]:
# Simulate what the vector store holds
knowledge_chunks = [
    "High-protein foods include chicken breast, turkey, lean beef, fish, eggs, Greek yogurt, cottage cheese, lentils, chickpeas, tofu, and edamame.",
    "Iron-rich foods: heme iron from red meat, liver, chicken, fish; non-heme iron from lentils, spinach, tofu. Pair with vitamin C to improve absorption.",
    "Omega-3 fatty acids found in salmon, walnuts, and flaxseed reduce inflammation and support heart and brain health.",
    "Strength training requires progressive overload — gradually increasing weight, reps, or sets over time to continue making gains.",
    "Adults need 7-9 hours of sleep per night. Sleep deprivation increases risk of obesity, diabetes, and cardiovascular disease.",
    "Mindfulness meditation for 10 minutes per day reduces cortisol and improves focus and stress resilience.",
    "Muscle hypertrophy requires 6-12 reps per set, 3-5 sets per exercise, with 10-20 total weekly sets per muscle group.",
    "Vitamin D supports bone health, immune function, and mood. It is synthesised from sunlight and found in fatty fish and egg yolks.",
]

chunk_embeddings = model.encode(knowledge_chunks, normalize_embeddings=True)

# Now query
test_query = "What should I eat to build muscle?"
q_emb = model.encode([test_query], normalize_embeddings=True)
scores = cosine_similarity(q_emb, chunk_embeddings)[0]
top_k = np.argsort(scores)[::-1][:3]

print(f'Query: "{test_query}"')
print(f'\nTop 3 retrieved chunks:\n')
for rank, idx in enumerate(top_k, 1):
    print(f'{rank}. [score={scores[idx]:.3f}] {knowledge_chunks[idx]}')
    print()

## Summary

You have now seen:
- Text is converted to a dense vector (384 numbers) by `all-MiniLM-L6-v2`
- Cosine similarity measures semantic closeness — not keyword overlap
- Health topics cluster in embedding space
- Given a query, we can rank chunks by similarity to find the most relevant ones

**Next: Phase 2** — store these embeddings in ChromaDB so retrieval persists across sessions.